# 03 – Gold: fact_purchases (one row per PO line)
stg_purchase_order_detail JOIN stg_purchase_order_header ON PurchaseOrderID → adds VendorID and OrderDate.
Saved next to `dim_vendor`. With `fact_sales`, `dim_product`, `dim_sales_territory` this gives 2 stars sharing `dim_product`.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
TARGET_SCHEMA = "silver"        # same schema as dim_vendor / fact_sales (change to "gold" if you keep facts there)
spark.sql(f"USE CATALOG {CATALOG}")

d = spark.table(f"{CATALOG}.silver.stg_purchase_order_detail").alias("d")
h = spark.table(f"{CATALOG}.silver.stg_purchase_order_header").alias("h")

fact_purchases = (d.join(h, "PurchaseOrderID", "inner")
    .select("PurchaseOrderID", "PurchaseOrderDetailID",
            F.col("h.VendorID").alias("VendorID"),
            F.col("h.OrderDate").alias("OrderDate"),
            "ProductID", "OrderQty", "UnitPrice", "LineTotal",
            "ReceivedQty", "RejectedQty", "StockedQty", "DueDate"))

fact_purchases.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{TARGET_SCHEMA}.fact_purchases")

print("fact_purchases rows:", fact_purchases.count())
display(fact_purchases.limit(10))

fact_purchases rows: 8845


PurchaseOrderID,PurchaseOrderDetailID,VendorID,OrderDate,ProductID,OrderQty,UnitPrice,LineTotal,ReceivedQty,RejectedQty,StockedQty,DueDate
1,1,1580,2022-04-15T00:00:00.000Z,1,4,50.2600,201.0400,3.00,0.00,3.00,2022-04-29T00:00:00.000Z
2,2,1496,2022-04-15T00:00:00.000Z,359,3,45.1200,135.3600,3.00,0.00,3.00,2022-04-29T00:00:00.000Z
2,3,1496,2022-04-15T00:00:00.000Z,360,3,45.5805,136.7415,3.00,0.00,3.00,2022-04-29T00:00:00.000Z
3,4,1494,2022-04-15T00:00:00.000Z,530,550,16.0860,8847.3000,550.00,0.00,550.00,2022-04-29T00:00:00.000Z
4,5,1650,2022-04-15T00:00:00.000Z,4,3,57.0255,171.0765,2.00,1.00,1.00,2022-04-29T00:00:00.000Z
5,6,1654,2022-04-29T00:00:00.000Z,512,550,37.0860,20397.3000,550.00,0.00,550.00,2022-05-13T00:00:00.000Z
6,7,1664,2022-04-29T00:00:00.000Z,513,550,26.5965,14628.0750,468.00,0.00,468.00,2022-05-13T00:00:00.000Z
7,8,1678,2022-04-29T00:00:00.000Z,317,550,27.0585,14882.1750,550.00,0.00,550.00,2022-05-13T00:00:00.000Z
7,9,1678,2022-04-29T00:00:00.000Z,318,550,33.5790,18468.4500,550.00,0.00,550.00,2022-05-13T00:00:00.000Z
7,10,1678,2022-04-29T00:00:00.000Z,319,550,46.0635,25334.9250,550.00,0.00,550.00,2022-05-13T00:00:00.000Z
